# OpenStreetMap amenities with browser-safe fallback

This notebook shows an OpenStreetMap workflow using Overpass API logic and a local fallback. In JupyterLite, network access can be blocked by CORS or temporary service limits, so a fallback pattern is part of the lesson.

The map emphasizes reproducibility: save query text, bounding box, date, and tags.

**Reflection questions:** What OSM tags would you trust for public-health analysis? How should volunteers and agencies document edits? What bias could appear in volunteered geographic information?

In [ ]:
# Pyodide/JupyterLite bootstrap: install only pure-Python packages used in this notebook.
import sys, importlib
try:
    import micropip
except Exception:
    micropip = None

async def ensure_packages(packages):
    for pkg, import_name in packages:
        try:
            importlib.import_module(import_name)
        except Exception:
            if micropip is None:
                raise RuntimeError(f'{pkg} is not installed and micropip is unavailable.')
            await micropip.install(pkg)

await ensure_packages([('pandas','pandas'), ('folium','folium'), ('branca','branca'), ('plotly','plotly')])


In [ ]:
from pathlib import Path
import json, math, statistics
import pandas as pd
import folium
from folium.plugins import MarkerCluster, HeatMap, TimestampedGeoJson, MiniMap, Fullscreen, MeasureControl

DATA = Path('../data')

def load_json(name):
    return json.loads((DATA / name).read_text(encoding='utf-8'))

def load_csv(name):
    return pd.read_csv(DATA / name)

def add_standard_controls(m):
    MiniMap(toggle_display=True).add_to(m)
    Fullscreen().add_to(m)
    MeasureControl(primary_length_unit='kilometers').add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m

def color_scale(values, colors=('green','orange','red')):
    vals = list(values)
    lo, hi = min(vals), max(vals)
    def pick(v):
        if hi == lo:
            return colors[1]
        t = (v - lo) / (hi - lo)
        return colors[0] if t < .33 else colors[1] if t < .66 else colors[2]
    return pick


In [ ]:
fac = load_csv('health_facilities_training_points.csv')
# Query text learners can paste into overpass-turbo.eu for live OSM exploration.
overpass_query = '''
[out:json][timeout:25];
(
  node["amenity"="hospital"](45.45,-73.70,45.58,-73.50);
  way["amenity"="hospital"](45.45,-73.70,45.58,-73.50);
  relation["amenity"="hospital"](45.45,-73.70,45.58,-73.50);
);
out center tags;
'''
print(overpass_query)
montreal = fac[fac.city.eq('Montreal')]
m = folium.Map(location=[45.51,-73.58], zoom_start=12, tiles='OpenStreetMap')
for _, r in montreal.iterrows():
    folium.Marker([r.lat, r.lon], tooltip=r['name'], popup=f"{r['name']}<br>fallback facility layer").add_to(m)
add_standard_controls(m)
m